# 数据会骗人：辛普森悖论、伯克森悖论、幸存者偏差与均值回归

本笔记本由文章自动生成，代码和[原文](https://wxrkz123.github.io/blog/Statistical-Paradoxes/)完全一致。文中的解释、公式、图和每段代码的输出，请对照原文阅读。

所有代码只用到 NumPy 和 pandas：

In [ ]:
import numpy as np
import pandas as pd

### 伯克利的招生数据

1973 年秋季，加州大学伯克利分校研究生院的录取数据显示，男生的录取率明显高于女生。学校担心被起诉性别歧视，统计学家 Bickel 等人（1975）仔细分析了这份数据。下面是申请人数最多的 6 个系（为了保密，系名用字母代替）：

In [ ]:
ucb = pd.DataFrame(
    [("A", 512, 825, 89, 108), ("B", 353, 560, 17, 25), ("C", 120, 325, 202, 593),
     ("D", 138, 417, 131, 375), ("E", 53, 191, 94, 393), ("F", 22, 373, 24, 341)],
    columns=["系", "男录取", "男申请", "女录取", "女申请"]).set_index("系")
ucb["男录取率"] = ucb["男录取"] / ucb["男申请"]
ucb["女录取率"] = ucb["女录取"] / ucb["女申请"]
print(ucb[["男申请", "男录取率", "女申请", "女录取率"]].round(2).to_string())
total = ucb.sum()
print(f"合计：男 {total['男录取'] / total['男申请']:.1%}，女 {total['女录取'] / total['女申请']:.1%}")

原因在申请人数那两列：**男生大多申请录取率高的系，女生大多申请录取率低的系**。

In [ ]:
ucb["系录取率"] = (ucb["男录取"] + ucb["女录取"]) / (ucb["男申请"] + ucb["女申请"])
easy = ucb.index[:2]                              # A、B 两系的录取率超过 60%
print(f"A、B 两系的总录取率：{ucb.loc[easy, '系录取率'].round(2).tolist()}")
print(f"申请 A、B 两系的比例：男 {ucb.loc[easy, '男申请'].sum() / total['男申请']:.0%}，"
      f"女 {ucb.loc[easy, '女申请'].sum() / total['女申请']:.0%}")

# 假设女生和男生一样分布在各系（用男生的申请分布加权），女生的录取率是多少？
w = ucb["男申请"] / total["男申请"]
print(f"按男生的申请分布加权：男 {(w * ucb['男录取率']).sum():.1%}，女 {(w * ucb['女录取率']).sum():.1%}")

### 一个连续变量的例子

同样的现象也会出现在回归中。假设我们调查了三个年龄段的人每周的运动时间和胆固醇水平：年龄越大，胆固醇越高，而年纪大的人也恰好运动更多；在每个年龄段内部，运动越多，胆固醇越低。

In [ ]:
rng = np.random.default_rng(1)
groups = []
for k, age in enumerate([30, 45, 60]):
    x = rng.normal(2 + 2 * k, 0.8, 200)                           # 每周运动小时数，年纪大的运动更多
    y = 150 + 1.2 * age - 6 * (x - (2 + 2 * k)) + rng.normal(0, 8, 200)
    groups.append(pd.DataFrame({"年龄": age, "运动": x, "胆固醇": y}))
health = pd.concat(groups, ignore_index=True)

slope = lambda d: np.polyfit(d["运动"], d["胆固醇"], 1)[0]
print(f"所有人一起回归：运动每多 1 小时，胆固醇 {slope(health):+.1f}")
for age, d in health.groupby("年龄"):
    print(f"  {age} 岁左右：{slope(d):+.1f}")

### 好论文为什么总是“要么严谨，要么有趣”？

用一个简单的模拟检验这个想法：一万篇投稿，严谨程度和新颖程度**完全独立**；审稿人看的是两者之和，超过门槛才能发表：

In [ ]:
rng = np.random.default_rng(2)
rigor, novelty = rng.normal(size=10_000), rng.normal(size=10_000)
published = rigor + novelty > 2
print(f"发表率：{published.mean():.1%}")
print(f"全部投稿中，严谨与新颖的相关系数：{np.corrcoef(rigor, novelty)[0, 1]:+.2f}")
print(f"已发表的论文中：{np.corrcoef(rigor[published], novelty[published])[0, 1]:+.2f}")

### 基金的业绩

模拟 2000 只基金，每只基金每年相对于市场的超额收益是均值为 0、标准差 10% 的随机数，也就是说，**没有任何基金经理有真正的选股能力**；累计超额收益跌破 -15% 的基金当年清盘：

In [ ]:
rng = np.random.default_rng(3)
n_funds, n_years = 2000, 10
excess = rng.normal(0, 0.10, (n_funds, n_years))      # 每年的超额收益，真实均值为 0
alive = np.ones(n_funds, dtype=bool)
life = np.full(n_funds, n_years)                     # 每只基金存续了几年
cum = np.zeros(n_funds)
for t in range(n_years):
    cum += np.where(alive, excess[:, t], 0)
    closed = alive & (cum < -0.15)
    life[closed] = t + 1
    alive &= ~closed

observed = [excess[i, :life[i]].mean() for i in range(n_funds)]   # 每只基金存续期间的平均超额收益
print(f"10 年后还存在的基金：{alive.mean():.0%}")
print(f"只看存活的基金：平均每年超额收益 {excess[alive].mean():+.2%}")
print(f"包括清盘的基金（各自存续期间）：{np.mean(observed):+.2%}")

### 补习班的“效果”

一个班 1000 个学生，每个学生的真实水平不同；每次考试的成绩等于真实水平加上运气（当天的状态、题目是否正好复习到）。第一次考试后，学校给成绩最差的 10% 安排了补习，第二次考试他们平均提高了很多。补习有效吗？

In [ ]:
rng = np.random.default_rng(4)
ability = rng.normal(70, 8, 1000)                  # 真实水平
test1 = ability + rng.normal(0, 6, 1000)           # 两次考试的运气相互独立
test2 = ability + rng.normal(0, 6, 1000)           # 第二次考试前，没有人做任何事
low, high = test1 <= np.quantile(test1, 0.1), test1 >= np.quantile(test1, 0.9)
print(f"第一次最差的 10%：{test1[low].mean():.1f} → {test2[low].mean():.1f}")
print(f"第一次最好的 10%：{test1[high].mean():.1f} → {test2[high].mean():.1f}")
print(f"用第一次成绩预测第二次的斜率：{np.polyfit(test1, test2, 1)[0]:.2f}")